In [ ]:
import geopandas as gpd
import pandas as pd

# Cargar parroquias rurales y urbanas
gdf_rurales = gpd.read_file("data/parroquiasRurales.geojson").rename(
    columns={"DPA_DESPAR": "nombre"}
)
gdf_urbanas = gpd.read_file("data/parroquiasUrbanas.geojson").rename(
    columns={"dpa_despar": "nombre"}
)

gdf_rurales["tipo"] = "rural"
gdf_urbanas["tipo"] = "urbana"

# Unir en un solo GeoDataFrame
gdf_parroquias = pd.concat(
    [
        gdf_rurales[["nombre", "geometry", "tipo"]],
        gdf_urbanas[["nombre", "geometry", "tipo"]],
    ],
    ignore_index=True,
).set_crs("EPSG:4326")

# Proyectar a CRS métrico para trabajar en metros
gdf_parroquias_m = gdf_parroquias.to_crs(epsg=32717)
gdf_parroquias_m["area_m2"] = gdf_parroquias_m.geometry.area

# Calcular estadísticas descriptivas
stats = gdf_parroquias_m["area_m2"].describe()

# Mostrar resultados con nombres claros y valores legibles
print("\n===== Estadísticas de Área de Parroquias (en m²) =====")
print(f"🔢 Total de parroquias       : {int(stats['count'])}")
print(f"📏 Área mínima              : {stats['min']:,.2f} m²")
print(f"📏 Primer cuartil (25%)     : {stats['25%']:,.2f} m²")
print(f"📏 Mediana (50%)            : {stats['50%']:,.2f} m²")
print(f"📏 Tercer cuartil (75%)     : {stats['75%']:,.2f} m²")
print(f"📏 Área máxima              : {stats['max']:,.2f} m²")
print(f"📐 Promedio (media)         : {stats['mean']:,.2f} m²")
print(f"📊 Desviación estándar      : {stats['std']:,.2f} m²")

# Calculo en m2
lado_celda = stats["50%"] ** 0.5
print(f"\n📦 Tamaño celda : {lado_celda:,.2f} m x {lado_celda:,.2f} m")


===== Estadísticas de Área de Parroquias (en m²) =====
🔢 Total de parroquias       : 65
📏 Área mínima              : 2,416,683.84 m²
📏 Primer cuartil (25%)     : 7,546,966.57 m²
📏 Mediana (50%)            : 17,695,184.03 m²
📏 Tercer cuartil (75%)     : 70,279,566.28 m²
📏 Área máxima              : 541,928,460.57 m²
📐 Promedio (media)         : 64,400,613.77 m²
📊 Desviación estándar      : 110,162,837.64 m²

📦 Tamaño sugerido de celda : 4,206.56 m x 4,206.56 m
